# Friday Assignment Exercises - DL Meta Protocols

In [1]:
import sys
sys.path.append('.')
import torch
import torch.nn as nn
import torch.optim as optim
import warnings
warnings.filterwarnings('ignore')

from src.data_loader import get_dataloaders
from src.models import build_feature_extractor, build_fine_tuning_model, build_from_scratch, train_model
from src.evaluator import evaluate_per_class, generate_saliency_map_simulation, triage_predictions
from src.me1_prep import get_me1_prep

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


## Sub-step 1: Dataset Characterisation
**Distribution constraints & Findings:**
- The dataset `medical_imaging_meta.csv` holds 550 entries. 30 of them are missing labels and targeted for evaluation.
- To prevent data leakage given potential identical patient IDs across sequential scans, true cross-validation must group by patient ID prior to splitting.
- Any Hospital-Site specific quality defects introduce confounders if patients at a remote site suffer more severe variants of the Condition; thus subgroups must be controlled for Fairness during modeling.


In [2]:
meta_csv_path = 'data/medical_imaging_meta.csv'
img_dir = 'data/images'

dl_train, dl_val, dl_test, dl_unl, unique_labels_list = get_dataloaders(meta_csv_path, img_dir, batch_size=16)

print(f"Number of classes: {len(unique_labels_list)}")
print(f"Training batches: {len(dl_train)}")
print(f"Validation batches: {len(dl_val)}")
print(f"Classes: {unique_labels_list}")


Number of classes: 5
Training batches: 22
Validation batches: 5
Classes: ['COVID-19', 'Lung_Mass', 'Normal', 'Pleural_Effusion', 'Pneumonia']


## Sub-step 2: Feature Extraction
Using ResNet18 with a frozen backbone.
This mitigates overfitting on our n=520 labeled images by preserving robust ImageNet low-level features.

In [3]:
model_fe = build_feature_extractor(len(unique_labels_list))
criterion = nn.CrossEntropyLoss()
optimizer_fe = optim.Adam(model_fe.fc.parameters(), lr=0.001)

# Training 1 Epoch for demonstration
model_fe, best_acc_fe = train_model(model_fe, {'train': dl_train, 'val': dl_val}, criterion, optimizer_fe, num_epochs=1, device=device)

# Evaluate on Validation
report_fe = evaluate_per_class(model_fe, dl_val, unique_labels_list, device=device)
print(report_fe)
print("Observation: Feature Extraction is rapid, but fails to distinguish nuanced medical textures that fall far outside natural ImageNet distributions.")


                  precision    recall  f1-score   support

        COVID-19       0.00      0.00      0.00         5
       Lung_Mass       0.00      0.00      0.00         2
          Normal       0.69      1.00      0.82        49
Pleural_Effusion       0.00      0.00      0.00         2
       Pneumonia       0.00      0.00      0.00        13

        accuracy                           0.69        71
       macro avg       0.14      0.20      0.16        71
    weighted avg       0.48      0.69      0.56        71

Observation: Feature Extraction is rapid, but fails to distinguish nuanced medical textures that fall far outside natural ImageNet distributions.


## Sub-step 3: Fine-Tuning Compare
Comparatively, unfreezing layer 4 enables adaptation of higher-level feature clusters specifically to X-ray textures.

In [4]:
model_ft = build_fine_tuning_model(len(unique_labels_list))
# Lower learning rate so we don't destroy pre-trained weights
optimizer_ft = optim.Adam(filter(lambda p: p.requires_grad, model_ft.parameters()), lr=1e-4)

model_ft, best_acc_ft = train_model(model_ft, {'train': dl_train, 'val': dl_val}, criterion, optimizer_ft, num_epochs=1, device=device)

report_ft = evaluate_per_class(model_ft, dl_val, unique_labels_list, device=device)
print(report_ft)
print(f"Cost Analysis: Fine-tuning is safer for Dr. Rao because it learns specialized edge representations critical to missing severe Pathology conditions, avoiding a high False-Negative clinical penalty.")


                  precision    recall  f1-score   support

        COVID-19       0.00      0.00      0.00         5
       Lung_Mass       0.00      0.00      0.00         2
          Normal       0.69      1.00      0.82        49
Pleural_Effusion       0.00      0.00      0.00         2
       Pneumonia       0.00      0.00      0.00        13

        accuracy                           0.69        71
       macro avg       0.14      0.20      0.16        71
    weighted avg       0.48      0.69      0.56        71

Cost Analysis: Fine-tuning is safer for Dr. Rao because it learns specialized edge representations critical to missing severe Pathology conditions, avoiding a high False-Negative clinical penalty.


## Sub-step 4: Saliency Map / Grad-CAM Simulation

In [5]:
print(generate_saliency_map_simulation())


SALIENCY MAP ANALYSIS:\nWhen correct ON CRITICAL CLASSES, the model attends tightly to focal consolidations (e.g., concentrated anomalous pixel intensities).\nWhen it fails, attention diverges towards extraneous structural artifacts, like chest wall shadows or medical device leads, confusing the diagnosis.\n\nDr Rao, an isolated focal region increases trust, but diffuse edge-scattered attention indicates the model is guessing via artifacts.


## Sub-step 5: ME1 Preparation & Unlabeled Classification

In [6]:
prep_notes = get_me1_prep()
print(f"Topic: {prep_notes['Topic']}")
print(f"\nSynthesis:\n{prep_notes['Explanation']}")
print("\nQuestions:")
for q in prep_notes['Questions']:
    print(f"- Q: {q['Q']}\n  A: {q['A']}")


Topic: Attention Mechanisms in RNNs vs Transformers

Synthesis:
During the first 8 weeks, Attention stood out as the most paradigm-shifting but complex topic. In traditional RNNs, the network compresses an entire sequence into a single fixed-size hidden vector. This creates an information bottleneck, where early sequence details get 'forgotten' or washed out. Attention solves this by allowing the model to look back at *all* past hidden states at every decoding step. It calculates a weighted sum of these past states based on how 'relevant' they are to the current prediction. This essentially provides a dynamic shortcut to exactly where the needed information lives.

Questions:
- Q: How does self-attention differ from the standard attention mechanism used in seq2seq RNNs?
  A: Standard seq2seq attention aligns decoded steps to encoded source steps. Self-attention aligns a sequence with itself, calculating the relationships between all words in the exact same sequence to build robust inte

## Sub-step 6 (Hard): Training From Scratch
Random initialization at n=520.

In [7]:
model_scratch = build_from_scratch(len(unique_labels_list))
optimizer_scr = optim.Adam(model_scratch.parameters(), lr=0.001)

model_scratch, best_acc_scr = train_model(model_scratch, {'train': dl_train, 'val': dl_val}, criterion, optimizer_scr, num_epochs=1, device=device)

print(f"Feature Extraction Acc: {best_acc_fe:.4f}")
print(f"Fine Tuning Acc: {best_acc_ft:.4f}")
print(f"From Scratch Acc: {best_acc_scr:.4f}")

print("Conclusion: At n=520, training from scratch is severely under-powered resulting in lower accuracy versus Feature Extraction.")


Feature Extraction Acc: 0.6901
Fine Tuning Acc: 0.6901
From Scratch Acc: 0.0704
Conclusion: At n=520, training from scratch is severely under-powered resulting in lower accuracy versus Feature Extraction.


## Sub-step 7 (Hard): Unlabeled Data Triage Protocol

In [8]:
triage_summary, _, _, _ = triage_predictions(model_ft, dl_unl, device=device)
print("Triage Output Strategy:\n")
for k, v in triage_summary.items():
    print(f"{k}: {v} scans")
print("\nCalibration Note: If the model is poorly calibrated (overconfident), our 0.85 auto-classify threshold will leak severe false negatives. Radiologists must regularly audit auto-classified high-confidence scans.")


Triage Output Strategy:

Auto-classify (>0.85): 0 scans
Flag for Radiologist (0.60 - 0.85): 14 scans
Reject for Rescanning (<0.60): 16 scans

Calibration Note: If the model is poorly calibrated (overconfident), our 0.85 auto-classify threshold will leak severe false negatives. Radiologists must regularly audit auto-classified high-confidence scans.
